# Pythia-410m Deep Convergence Probe

**Date:** 2026-03-25  
**Depends on:** Stage 1 results (01_attractor_dominance.ipynb)  
**Outputs:** `output_deep/` directory

## Purpose

Stage 1 showed Pythia-410m producing **30+ scattered terminal tokens** at 100 iterations —
no convergent attractor basins. This probe extends to **1000 iterations** on a
representative 25-prompt subset to determine whether:

- **(a)** The model was under-iterated and convergence appears at higher iteration counts
- **(b)** The latent manifold is genuinely flat with no discrete attractors at this scale

### Design

| Parameter | Value |
|:---|:---|
| Iterations | 1000 (4× Stage 1) |
| Schedule | `[0, 5, 10, 25, 50, 100, 200, 300, 500, 750, 1000]` |
| Prompts | 25 (subset spanning all 7 categories) |
| Model | `pythia-410m` (24L, 1024d) |
| Layers | 0 → 23 (full depth) |

---

In [ ]:
# ============================================================
# STEP 0: DEPENDENCIES
# ============================================================
import sys
!{sys.executable} -m pip install kaleido -q

In [ ]:
# --- harness bootstrap: neutralise kaleido write_image ---
import plotly.basedatatypes as _bdt
def _skip_write_image(self, *args, **kwargs):
    _p = args[0] if args else kwargs.get('file', '<figure>')
    print(f'[write_image skipped — kaleido disabled on this host] {_p}')
_bdt.BaseFigure.write_image = _skip_write_image
print('write_image neutralised; charts still render via fig.show()')


In [ ]:
# ============================================================
# STEP 1: MODEL SETUP
# ============================================================
import torch
import numpy as np
import os
import plotly.graph_objects as go
import plotly.express as px
from transformer_lens import HookedTransformer
from IPython.display import Markdown, display
import warnings
warnings.filterwarnings('ignore')

device = "cuda" if torch.cuda.is_available() else "cpu"
model = HookedTransformer.from_pretrained("pythia-410m", device=device)
print(f"Architecture: {model.cfg.n_layers} layers, {model.cfg.n_heads} heads, d_model={model.cfg.d_model}")
print(f"Running on: {device}")

OUTPUT_DIR = "output_deep"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {os.path.abspath(OUTPUT_DIR)}")

In [ ]:
# ============================================================
# STEP 2: PROMPT SUBSET (25 from 125)
# ============================================================
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join('.', '..', '..')))

from prompt_library import (
    COMPLEX, NARRATIVE, SIMPLE, CHEMICAL, ACRONYMS, VULGARITY, WILD,
    CATEGORY_MAP
)

# 25 prompts: 4 each from Complex/Narrative/Simple/Wild, 3 each from Chemical/Acronyms/Vulgarity
DEEP_SUBSET_KEYS = [
    # Complex (4)
    "A01_physics", "A06_epistemology", "A09_code", "A17_marx",
    # Narrative (4)
    "B01_napoleon", "B05_mlk", "B10_weather", "B17_argument",
    # Simple (4)
    "C01_jack_jill", "C05_twinkle", "C11_genesis", "C18_wolf",
    # Chemical (3)
    "D01_water", "D04_equation", "D08_math",
    # Acronyms (3)
    "E01_politics", "E03_orgs", "E07_military",
    # Vulgarity (3)
    "F01_anger", "F04_argument", "F09_slur_adjacent",
    # Wild (4)
    "G01_punctuation", "G07_the", "G13_buffalo", "G23_emoji",
]

# Build the subset library from the full categories
ALL_CATS = {**COMPLEX, **NARRATIVE, **SIMPLE, **CHEMICAL, **ACRONYMS, **VULGARITY, **WILD}
PROMPT_SUBSET = {k: ALL_CATS[k] for k in DEEP_SUBSET_KEYS}

# Extended schedule for deep probing
ITERATION_SCHEDULE = [0, 5, 10, 25, 50, 100, 200, 300, 500, 750, 1000]
MAX_ITERATIONS = max(ITERATION_SCHEDULE)

LAYER_START = 0
LAYER_END = model.cfg.n_layers - 1

print(f"Schedule: {ITERATION_SCHEDULE}")
print(f"Room: Layers {LAYER_START} -> {LAYER_END}")
print(f"Prompts: {len(PROMPT_SUBSET)} (subset of 125)")
print(f"Max iterations: {MAX_ITERATIONS}")

# Category breakdown
cat_counts = {}
for k in DEEP_SUBSET_KEYS:
    cat = CATEGORY_MAP[k]
    cat_counts[cat] = cat_counts.get(cat, 0) + 1
for cat, count in cat_counts.items():
    print(f"  {cat}: {count}")

# Save config
config_md = f"""# Deep Convergence Probe Config
- **Model:** pythia-410m
- **Prompts:** {len(PROMPT_SUBSET)} (subset)
- **Schedule:** {ITERATION_SCHEDULE}
- **Max Iterations:** {MAX_ITERATIONS}
- **Layers:** {LAYER_START} -> {LAYER_END}
- **Device:** {device}
"""
with open(os.path.join(OUTPUT_DIR, 'config.md'), 'w', encoding='utf-8') as f:
    f.write(config_md)
print(f"\n[SAVED] {OUTPUT_DIR}/config.md")

In [ ]:
# ============================================================
# STEP 3: ATR ENGINE (inline, identical to Stage 1)
# ============================================================

def get_top_tokens(model, resid_vector, k=5):
    """Decode a residual stream vector into top-k token predictions.
    Applies the Final LayerNorm before unembedding for correct decoding."""
    normalized = model.ln_final(resid_vector)
    logits = normalized @ model.W_U + model.b_U
    probs = torch.softmax(logits, dim=-1)
    top_probs, top_indices = torch.topk(probs, k)
    tokens = [model.tokenizer.decode([idx]) for idx in top_indices]
    return list(zip(tokens, top_probs.tolist()))


def run_total_resonance_loop(model, prompt, layer_start, layer_end, max_iter, schedule):
    """
    TOTAL Lucier Loop: iteratively re-inject the ENTIRE residual stream
    tensor (all token positions) through the layer slice.
    Returns a list of snapshot dicts at each scheduled iteration.
    """
    snapshots = []
    hook_point_read = f"blocks.{layer_end}.hook_resid_post"
    hook_point_write = f"blocks.{layer_start}.hook_resid_pre"
    
    with torch.no_grad():
        _, cache = model.run_with_cache(
            prompt,
            names_filter=lambda n: n == hook_point_read
        )
    
    current_tensor = cache[hook_point_read][0].clone()
    seq_len = current_tensor.shape[0]
    initial_norm = current_tensor.norm().item()
    
    last_vec = current_tensor[-1, :].clone()
    mean_vec = current_tensor.mean(dim=0).clone()
    
    if 0 in schedule:
        top_tokens_last = get_top_tokens(model, last_vec)
        all_pos_tokens = []
        for pos in range(seq_len):
            pos_top = get_top_tokens(model, current_tensor[pos, :], k=1)
            all_pos_tokens.append(pos_top[0][0])
        snapshots.append({
            "iteration": 0,
            "tensor": current_tensor.clone().cpu(),
            "last_vector": last_vec.clone().cpu(),
            "mean_vector": mean_vec.clone().cpu(),
            "last_norm": last_vec.norm().item(),
            "mean_norm": mean_vec.norm().item(),
            "tensor_norm": current_tensor.norm().item(),
            "top_tokens": top_tokens_last,
            "all_position_tokens": all_pos_tokens,
            "cosine_sim_last": 1.0,
            "cosine_sim_mean": 1.0,
            "position_similarity": 1.0,
        })
    
    prev_last = last_vec.clone()
    prev_mean = mean_vec.clone()
    
    for i in range(1, max_iter + 1):
        # Normalise to maintain energy level
        current_norm = current_tensor.norm().item()
        if current_norm > 0:
            current_tensor = current_tensor * (initial_norm / current_norm)
        
        inject_tensor = current_tensor.clone()
        
        def injection_hook(resid, hook, tensor=inject_tensor):
            resid[0, :, :] = tensor
            return resid
        
        model.add_hook(hook_point_write, injection_hook)
        try:
            with torch.no_grad():
                _, cache = model.run_with_cache(
                    prompt,
                    names_filter=lambda n: n == hook_point_read
                )
        finally:
            model.reset_hooks()
        
        current_tensor = cache[hook_point_read][0].clone()
        last_vec = current_tensor[-1, :].clone()
        mean_vec = current_tensor.mean(dim=0).clone()
        
        if i in schedule:
            cos_sim_last = torch.nn.functional.cosine_similarity(
                last_vec.unsqueeze(0), prev_last.unsqueeze(0)
            ).item()
            cos_sim_mean = torch.nn.functional.cosine_similarity(
                mean_vec.unsqueeze(0), prev_mean.unsqueeze(0)
            ).item()
            
            pos_norms = current_tensor.norm(dim=1, keepdim=True).clamp(min=1e-8)
            normalized_positions = current_tensor / pos_norms
            pos_sim_matrix = normalized_positions @ normalized_positions.T
            mask = ~torch.eye(seq_len, dtype=torch.bool, device=pos_sim_matrix.device)
            position_similarity = pos_sim_matrix[mask].mean().item()
            
            top_tokens_last = get_top_tokens(model, last_vec)
            all_pos_tokens = []
            for pos in range(seq_len):
                pos_top = get_top_tokens(model, current_tensor[pos, :], k=1)
                all_pos_tokens.append(pos_top[0][0])
            
            snapshots.append({
                "iteration": i,
                "tensor": current_tensor.clone().cpu(),
                "last_vector": last_vec.clone().cpu(),
                "mean_vector": mean_vec.clone().cpu(),
                "last_norm": last_vec.norm().item(),
                "mean_norm": mean_vec.norm().item(),
                "tensor_norm": current_tensor.norm().item(),
                "top_tokens": top_tokens_last,
                "all_position_tokens": all_pos_tokens,
                "cosine_sim_last": cos_sim_last,
                "cosine_sim_mean": cos_sim_mean,
                "position_similarity": position_similarity,
            })
            print(f"  iter {i:>4}: top='{top_tokens_last[0][0].strip()}', "
                  f"cos_mean={cos_sim_mean:.4f}, pos_collapse={position_similarity:.4f}")
        
        prev_last = last_vec.clone()
        prev_mean = mean_vec.clone()
    
    return snapshots

print("Engine loaded.")

In [ ]:
# ============================================================
# STEP 4: RUN ALL 25 PROMPTS (1000 iterations each)
# ============================================================

all_results = {}

for idx, (label, prompt) in enumerate(PROMPT_SUBSET.items()):
    print(f"\n{'='*60}")
    print(f"[{idx+1}/{len(PROMPT_SUBSET)}] RECORDING: '{label}'")
    print(f"  Prompt: \"{prompt}\"")
    print(f"{'='*60}")
    
    snapshots = run_total_resonance_loop(
        model, prompt,
        layer_start=LAYER_START,
        layer_end=LAYER_END,
        max_iter=MAX_ITERATIONS,
        schedule=ITERATION_SCHEDULE
    )
    all_results[label] = snapshots
    
    terminal = snapshots[-1]['top_tokens'][0][0].strip()
    print(f"  → Terminal token: '{terminal}'")

print(f"\n{'='*60}")
print(f"ALL {len(all_results)} RECORDINGS COMPLETE.")

---
## 5. Analysis

### 5a. Basin Assessment — Terminal States at 1000 Iterations

In [ ]:
# ============================================================
# VIS 5a: BASIN ASSESSMENT — Pure Observation
# ============================================================

md = "# Deep Convergence Probe: Basin Assessment (1000 iterations)\n\n"
md += "| Prompt | Category | Terminal Basin |\n"
md += "|:---|:---|:---|\n"

basin_counts = {}
category_basins = {}

for label, snapshots in all_results.items():
    terminal = snapshots[-1]['top_tokens'][0][0].strip()
    category = CATEGORY_MAP[label]
    
    basin_counts[terminal] = basin_counts.get(terminal, 0) + 1
    
    if category not in category_basins:
        category_basins[category] = []
    category_basins[category].append((label, terminal))
    
    md += f"| {label} | {category} | `{terminal}` |\n"

md += "\n---\n\n"
md += "## Basin Summary\n\n"
md += "| Basin | Count | % |\n"
md += "|:---|:---|:---|\n"
total = len(all_results)
for basin, count in sorted(basin_counts.items(), key=lambda x: -x[1]):
    md += f"| **{basin}** | {count} | {count/total*100:.1f}% |\n"

md += "\n---\n\n"
md += "## Category Breakdown\n\n"
for cat, entries in category_basins.items():
    md += f"### {cat} ({len(entries)} prompts)\n"
    cat_basins = {}
    for label, tok in entries:
        cat_basins[tok] = cat_basins.get(tok, 0) + 1
    for b, c in sorted(cat_basins.items(), key=lambda x: -x[1]):
        md += f"- {b}: {c}/{len(entries)}\n"
    md += "\n"

# Convergence vs Stage 1 comparison
n_unique = len(basin_counts)
md += f"## Convergence Assessment\n\n"
md += f"- **Unique terminal tokens at 1000 iters:** {n_unique}\n"
md += f"- **Stage 1 (100 iters) had:** 30+ scattered tokens\n"
if n_unique <= 5:
    md += f"- **Verdict:** Convergence observed — under-iteration confirmed\n"
elif n_unique <= 15:
    md += f"- **Verdict:** Partial convergence — some basin formation\n"
else:
    md += f"- **Verdict:** No convergence — manifold is genuinely flat\n"

with open(os.path.join(OUTPUT_DIR, 'basin_assessment.md'), 'w', encoding='utf-8') as f:
    f.write(md)
print(f"[SAVED] {OUTPUT_DIR}/basin_assessment.md")
display(Markdown(md))

### 5b. Cross-Prompt Convergence Matrix

In [ ]:
# ============================================================
# VIS 5b: CROSS-PROMPT CONVERGENCE MATRIX
# ============================================================

labels = list(all_results.keys())
n = len(labels)
sim_matrix = np.zeros((n, n))

final_vectors = []
for label in labels:
    final_vec = all_results[label][-1]["mean_vector"]
    final_vectors.append(final_vec)

for i in range(n):
    for j in range(n):
        sim_matrix[i, j] = torch.nn.functional.cosine_similarity(
            final_vectors[i].unsqueeze(0).float(),
            final_vectors[j].unsqueeze(0).float()
        ).item()

fig_sim = px.imshow(
    sim_matrix,
    x=labels, y=labels,
    color_continuous_scale="Viridis",
    title="Deep Convergence: Cross-Prompt Similarity at 1000 Iterations (25 Prompts)",
    aspect="auto",
)
fig_sim.update_layout(template="plotly_dark", height=700, width=900)
fig_sim.show()
fig_sim.write_image(os.path.join(OUTPUT_DIR, 'convergence_matrix.png'), scale=2)
print(f"[SAVED] {OUTPUT_DIR}/convergence_matrix.png")

off_diag = sim_matrix[np.triu_indices(n, k=1)]
print(f"\nMean cross-prompt similarity: {off_diag.mean():.4f}")
print(f"Min:  {off_diag.min():.4f}")
print(f"Max:  {off_diag.max():.4f}")

### 5c. Dissolution Pathways (Extended)

In [ ]:
# ============================================================
# VIS 5c: DISSOLUTION PATHWAYS — Extended to 1000 iters
# ============================================================

md = "# Deep Dissolution Pathways — Last-Token Top Prediction\n\n"

for cat_name, cat_dict in [
    ("Complex", COMPLEX), ("Narrative", NARRATIVE),
    ("Simple", SIMPLE), ("Chemical", CHEMICAL),
    ("Acronyms", ACRONYMS), ("Vulgarity", VULGARITY),
    ("Wild", WILD)
]:
    cat_labels = [k for k in DEEP_SUBSET_KEYS if k in cat_dict and k in all_results]
    if not cat_labels:
        continue
    
    md += f"## {cat_name} ({len(cat_labels)} prompts)\n\n"
    md += "| Iter | " + " | ".join(cat_labels) + " |\n"
    md += "| :--- | " + " | ".join([":---"] * len(cat_labels)) + " |\n"
    
    for idx, iteration in enumerate(ITERATION_SCHEDULE):
        row = f"| **{iteration}** |"
        for label in cat_labels:
            snapshots = all_results[label]
            if idx < len(snapshots):
                tok = snapshots[idx]['top_tokens'][0][0]
                clean_t = tok.replace('\n', '↵').replace('`', "'").strip()
                row += f" `{clean_t}` |"
            else:
                row += " — |"
        md += row + "\n"
    md += "\n"

with open(os.path.join(OUTPUT_DIR, 'dissolution_pathways.md'), 'w', encoding='utf-8') as f:
    f.write(md)
print(f"[SAVED] {OUTPUT_DIR}/dissolution_pathways.md")
display(Markdown(md))

### 5d. Convergence Dynamics — Phase Transition Detection

In [ ]:
# ============================================================
# VIS 5d: CONVERGENCE DYNAMICS — Does a phase transition occur?
# ============================================================
import pandas as pd

# Track how the number of unique terminal tokens changes across iterations
convergence_data = []

for sched_idx, iteration in enumerate(ITERATION_SCHEDULE):
    tokens_at_iter = set()
    for label, snapshots in all_results.items():
        if sched_idx < len(snapshots):
            tok = snapshots[sched_idx]['top_tokens'][0][0].strip()
            tokens_at_iter.add(tok)
    convergence_data.append({
        'Iteration': iteration,
        'Unique Tokens': len(tokens_at_iter),
    })

conv_df = pd.DataFrame(convergence_data)

fig_conv = go.Figure()
fig_conv.add_trace(go.Scatter(
    x=conv_df['Iteration'], y=conv_df['Unique Tokens'],
    mode='lines+markers',
    name='Unique Terminal Tokens',
    line=dict(width=3),
    marker=dict(size=10)
))
fig_conv.update_layout(
    template="plotly_dark",
    title="Convergence Dynamics: Unique Terminal Tokens vs Iteration",
    xaxis_title="Iteration",
    yaxis_title="Unique Terminal Tokens (out of 25)",
    height=500, width=900,
    xaxis_type="log",
)
fig_conv.show()
fig_conv.write_image(os.path.join(OUTPUT_DIR, 'convergence_dynamics.png'), scale=2)
print(f"[SAVED] {OUTPUT_DIR}/convergence_dynamics.png")

# Also track mean cosine similarity (stability metric)
stability_data = []
for sched_idx, iteration in enumerate(ITERATION_SCHEDULE):
    cos_sims = []
    for label, snapshots in all_results.items():
        if sched_idx < len(snapshots):
            cos_sims.append(snapshots[sched_idx]['cosine_sim_mean'])
    stability_data.append({
        'Iteration': iteration,
        'Mean Cosine Stability': np.mean(cos_sims) if cos_sims else 0,
        'Min Cosine Stability': np.min(cos_sims) if cos_sims else 0,
    })

stab_df = pd.DataFrame(stability_data)

fig_stab = go.Figure()
fig_stab.add_trace(go.Scatter(
    x=stab_df['Iteration'], y=stab_df['Mean Cosine Stability'],
    mode='lines+markers', name='Mean cos(iter_n, iter_n-1)',
    line=dict(width=3), marker=dict(size=10)
))
fig_stab.add_trace(go.Scatter(
    x=stab_df['Iteration'], y=stab_df['Min Cosine Stability'],
    mode='lines+markers', name='Min cos(iter_n, iter_n-1)',
    line=dict(width=2, dash='dash'), marker=dict(size=6)
))
fig_stab.update_layout(
    template="plotly_dark",
    title="Stability: Mean Cosine Similarity Between Consecutive Snapshots",
    xaxis_title="Iteration",
    yaxis_title="Cosine Similarity",
    height=500, width=900,
    xaxis_type="log",
    yaxis_range=[0.9, 1.001],
)
fig_stab.show()
fig_stab.write_image(os.path.join(OUTPUT_DIR, 'stability_dynamics.png'), scale=2)
print(f"[SAVED] {OUTPUT_DIR}/stability_dynamics.png")

### 5e. 3D PCA Trajectories — Extended

In [ ]:
# ============================================================
# VIS 5e: 3D PCA TRAJECTORIES
# ============================================================
from sklearn.decomposition import PCA

all_vecs = []
labels_list = []
cats_list = []
iters_list = []
text_list = []

for label, snapshots in all_results.items():
    for s in snapshots:
        all_vecs.append(s["mean_vector"].detach().cpu().numpy())
        labels_list.append(label)
        cats_list.append(CATEGORY_MAP.get(label, 'Unknown'))
        iters_list.append(s["iteration"])
        top_tok = s['top_tokens'][0][0].replace('\n', '↵').strip()
        text_list.append(f"Iter {s['iteration']}: {top_tok}")

all_vecs = np.array(all_vecs)
pca = PCA(n_components=3)
vecs_3d = pca.fit_transform(all_vecs)

df = pd.DataFrame({
    'x': vecs_3d[:, 0],
    'y': vecs_3d[:, 1],
    'z': vecs_3d[:, 2],
    'Prompt': labels_list,
    'Category': cats_list,
    'Iteration': iters_list,
    'Top_Token': text_list
})

fig_topo = px.line_3d(
    df, x='x', y='y', z='z',
    color='Category',
    hover_name='Top_Token',
    markers=True,
    title=f"Deep Convergence: Attractor Landscape — {len(all_results)} Prompts, 1000 iters<br>"
          f"<sup>(Explained Variance: {sum(pca.explained_variance_ratio_)*100:.1f}%)</sup>"
)
fig_topo.update_traces(marker=dict(size=3), line=dict(width=2))
fig_topo.update_layout(
    template="plotly_dark",
    height=900,
    width=1200,
    scene=dict(
        xaxis_title="PC 1",
        yaxis_title="PC 2",
        zaxis_title="PC 3",
    )
)
fig_topo.show()
fig_topo.write_image(os.path.join(OUTPUT_DIR, 'topology_3d.png'), scale=2)
print(f"[SAVED] {OUTPUT_DIR}/topology_3d.png")

### 5f. Basin Distribution

In [ ]:
# ============================================================
# VIS 5f: BASIN DISTRIBUTION BAR CHART
# ============================================================

basin_data = []
for label, snapshots in all_results.items():
    terminal = snapshots[-1]['top_tokens'][0][0].strip()
    category = CATEGORY_MAP.get(label, 'Unknown')
    basin_data.append({'Prompt': label, 'Category': category, 'Basin': terminal})

basin_df = pd.DataFrame(basin_data)
basin_summary = basin_df.groupby(['Category', 'Basin']).size().reset_index(name='Count')

fig_basin = px.bar(
    basin_summary, x='Category', y='Count', color='Basin',
    title=f"Deep Convergence: Basin Distribution by Category ({len(all_results)} prompts, 1000 iters)",
    barmode='stack'
)
fig_basin.update_layout(template="plotly_dark", height=500, width=900)
fig_basin.show()
fig_basin.write_image(os.path.join(OUTPUT_DIR, 'basin_distribution.png'), scale=2)
print(f"[SAVED] {OUTPUT_DIR}/basin_distribution.png")

In [ ]:
# ============================================================
# STEP 6: SAVE RAW DATA
# ============================================================

save_data = {}
for label, snapshots in all_results.items():
    save_data[label] = {
        "iterations": [s["iteration"] for s in snapshots],
        "last_vectors": torch.stack([s["last_vector"] for s in snapshots]),
        "mean_vectors": torch.stack([s["mean_vector"] for s in snapshots]),
        "last_norms": [s["last_norm"] for s in snapshots],
        "mean_norms": [s["mean_norm"] for s in snapshots],
        "cosine_sims_last": [s["cosine_sim_last"] for s in snapshots],
        "cosine_sims_mean": [s["cosine_sim_mean"] for s in snapshots],
        "position_similarity": [s["position_similarity"] for s in snapshots],
        "top_tokens": [s["top_tokens"] for s in snapshots],
        "all_position_tokens": [s["all_position_tokens"] for s in snapshots],
    }

torch.save(save_data, os.path.join(OUTPUT_DIR, 'deep_results.pt'))
print(f"[SAVED] {OUTPUT_DIR}/deep_results.pt")

config = {
    "schedule": ITERATION_SCHEDULE,
    "layer_start": LAYER_START,
    "layer_end": LAYER_END,
    "prompt_count": len(PROMPT_SUBSET),
    "prompt_keys": DEEP_SUBSET_KEYS,
    "model": "pythia-410m",
    "mode": "deep_convergence_probe",
    "max_iterations": MAX_ITERATIONS,
}
torch.save(config, os.path.join(OUTPUT_DIR, 'deep_config.pt'))
print(f"[SAVED] {OUTPUT_DIR}/deep_config.pt")
print(f"\n✅ All artifacts saved to {os.path.abspath(OUTPUT_DIR)}")